# 🔭 Module 11: LangSmith — Observability & Debugging

---

## What is LangSmith?

**LangSmith** is the observability platform for LangChain. It lets you:

- 👁️ **Trace** every LLM call, chain step, and tool invocation
- 🐛 **Debug** failures and unexpected outputs
- 📊 **Evaluate** model performance with datasets
- 🧪 **Test** prompts in the playground
- 📈 **Monitor** production latency and token usage
- 🏷️ **Tag and filter** runs for analysis

---

## Setup LangSmith

1. Sign up at [smith.langchain.com](https://smith.langchain.com)
2. Create an API key
3. Set environment variables:

```bash
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__your_key_here
LANGCHAIN_PROJECT=my-project-name
```

That's it! All LangChain calls are automatically traced.

---

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "LangChain-Mastery-Course"

from langchain_groq import ChatGroq, HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)
print("LangSmith tracing enabled ✅")
print(f"Project: {os.environ.get('LANGCHAIN_PROJECT', 'default')}")

## 1️⃣ Automatic Tracing

In [ ]:
# ============================================================
# Once LANGCHAIN_TRACING_V2=true is set,
# EVERY chain run is automatically sent to LangSmith!
# ============================================================

chain = (
    ChatPromptTemplate.from_template("Explain {concept} in simple terms.")
    | llm
    | StrOutputParser()
)

# This call is automatically traced!
result = chain.invoke({"concept": "quantum entanglement"})
print(result)
print("\n💡 Check smith.langchain.com to see this run traced!")

## 2️⃣ Manual Tracing with @traceable

In [ ]:
from langsmith import traceable

# ============================================================
# @traceable — trace any Python function in LangSmith!
# ============================================================

@traceable(run_type="chain", name="DataAnalysisPipeline")
def analyze_data(data: str, question: str) -> dict:
    """A complex multi-step analysis pipeline"""
    
    # Step 1: Extract key metrics
    metrics_chain = (
        ChatPromptTemplate.from_template(
            "Extract 3 key metrics from this data: {data}"
        )
        | llm | StrOutputParser()
    )
    metrics = metrics_chain.invoke({"data": data})
    
    # Step 2: Answer the question
    answer_chain = (
        ChatPromptTemplate.from_template(
            "Based on these metrics: {metrics}\n\nAnswer: {question}"
        )
        | llm | StrOutputParser()
    )
    answer = answer_chain.invoke({"metrics": metrics, "question": question})
    
    return {"metrics": metrics, "answer": answer}

result = analyze_data(
    data="Sales Q1: $1.2M, Q2: $1.5M, Q3: $2.1M, Q4: $1.8M. Employees: 45. Regions: 3.",
    question="What was the best performing quarter and by how much?"
)

print("Metrics:", result["metrics"][:100], "...")
print("\nAnswer:", result["answer"])

In [ ]:
# ============================================================
# Nested @traceable functions
# ============================================================

@traceable(name="preprocess")
def preprocess_text(text: str) -> str:
    """Clean and normalize text"""
    return text.lower().strip()

@traceable(name="classify")
def classify_text(text: str) -> str:
    """Classify the text category"""
    chain = (
        ChatPromptTemplate.from_template(
            "Classify this text into one category (tech/business/science/other):\n{text}\n\nCategory:"
        )
        | llm | StrOutputParser()
    )
    return chain.invoke({"text": text}).strip()

@traceable(name="TextProcessingPipeline")  # Parent span
def full_pipeline(raw_text: str) -> dict:
    """Full processing pipeline"""
    clean = preprocess_text(raw_text)    # Child span 1
    category = classify_text(clean)      # Child span 2
    return {"original": raw_text, "clean": clean, "category": category}

result = full_pipeline("LangChain Released LangGraph 0.2 with improved multi-agent support!")
print(f"Category: {result['category']}")
print("\n💡 LangSmith shows these as nested spans in a trace!")

## 3️⃣ Adding Metadata and Tags to Runs

In [ ]:
from langchain_core.runnables import RunnableConfig

# ============================================================
# Add metadata to any chain run for filtering in LangSmith
# ============================================================

config = RunnableConfig(
    tags=["production", "customer-support", "v2"],
    metadata={
        "user_id": "user_123",
        "session_id": "sess_abc",
        "environment": "production",
        "feature_flag": "new_prompt_v2"
    }
)

chain = (
    ChatPromptTemplate.from_template("Help with: {question}")
    | llm | StrOutputParser()
)

result = chain.invoke(
    {"question": "How do I reset my password?"},
    config=config  # Attaches metadata to LangSmith trace!
)

print(result)
print("\n💡 In LangSmith, you can filter by tags and metadata!")

## 4️⃣ Evaluation with LangSmith

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# ============================================================
# Create a dataset for evaluation
# ============================================================
client = Client()

# Define test cases
examples = [
    {
        "inputs": {"question": "What is 2 + 2?"},
        "outputs": {"answer": "4"}
    },
    {
        "inputs": {"question": "What is the capital of France?"},
        "outputs": {"answer": "Paris"}
    },
    {
        "inputs": {"question": "Who wrote Romeo and Juliet?"},
        "outputs": {"answer": "William Shakespeare"}
    },
]

# Create dataset in LangSmith
dataset_name = "langchain-mastery-qa-eval"

try:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Q&A evaluation dataset for LangChain mastery course"
    )
    
    # Add examples to dataset
    client.create_examples(
        inputs=[e["inputs"] for e in examples],
        outputs=[e["outputs"] for e in examples],
        dataset_id=dataset.id
    )
    print(f"Dataset created: {dataset_name} with {len(examples)} examples")
except Exception as e:
    print(f"Note: {e}")
    print("(Dataset may already exist or LangSmith key may not be set)")

In [ ]:
# ============================================================
# Define the system to evaluate
# ============================================================
def qa_system(inputs: dict) -> dict:
    """The system we want to evaluate"""
    chain = (
        ChatPromptTemplate.from_template("Answer this question concisely: {question}")
        | llm | StrOutputParser()
    )
    answer = chain.invoke(inputs)
    return {"answer": answer}

# ============================================================
# Run evaluation
# ============================================================
try:
    results = evaluate(
        qa_system,
        data=dataset_name,
        evaluators=[
            LangChainStringEvaluator("exact_match"),  # Exact string match
            LangChainStringEvaluator("embedding_distance"),  # Semantic similarity
        ],
        experiment_prefix="gpt-4o-mini-v1",
        metadata={"model": "llama-3.1-8b-instant", "temperature": 0}
    )
    
    print("\nEvaluation Results:")
    print(f"Total examples: {len(examples)}")
    print("View detailed results at: smith.langchain.com")
except Exception as e:
    print(f"Evaluation note: {e}")

## 5️⃣ Custom Evaluators

In [ ]:
# ============================================================
# Custom LLM-as-Judge evaluator
# ============================================================
from langchain_core.prompts import ChatPromptTemplate
from langsmith.evaluation import EvaluationResult

def helpfulness_evaluator(run, example) -> EvaluationResult:
    """
    Custom evaluator: Use LLM to judge if response is helpful
    Returns score from 0 to 1
    """
    judge_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    question = example.inputs.get("question", "")
    response = run.outputs.get("answer", "")
    expected = example.outputs.get("answer", "")
    
    judge_prompt = ChatPromptTemplate.from_template("""
    Question: {question}
    Expected Answer: {expected}
    Actual Answer: {response}
    
    Rate the accuracy of the actual answer on a scale of 0-10.
    Respond with ONLY a number (0-10).
    """)
    
    result = judge_llm.invoke(
        judge_prompt.format(question=question, expected=expected, response=response)
    )
    
    try:
        score = float(result.content.strip()) / 10
    except:
        score = 0.5
    
    return EvaluationResult(
        key="accuracy",
        score=score,
        comment=f"LLM judge score: {score:.1f}"
    )

print("Custom evaluator defined ✅")
print("Usage:")
print("  results = evaluate(")
print("      qa_system,")
print("      data=dataset_name,")
print("      evaluators=[helpfulness_evaluator]")
print("  )")

## 6️⃣ Monitoring Production Metrics

In [ ]:
# ============================================================
# Query run data programmatically
# ============================================================
from langsmith import Client
from datetime import datetime, timedelta

client = Client()

try:
    # Get recent runs
    runs = list(client.list_runs(
        project_name="LangChain-Mastery-Course",
        start_time=datetime.now() - timedelta(hours=1),
        limit=10
    ))
    
    print(f"Recent runs (last hour): {len(runs)}")
    
    if runs:
        total_tokens = sum(r.total_tokens or 0 for r in runs)
        total_cost = sum(r.total_cost or 0 for r in runs)
        avg_latency = sum(r.latency or 0 for r in runs) / len(runs)
        errors = sum(1 for r in runs if r.error)
        
        print(f"\nMetrics Summary:")
        print(f"  Total tokens used: {total_tokens:,}")
        print(f"  Estimated cost: ${total_cost:.4f}")
        print(f"  Avg latency: {avg_latency:.2f}s")
        print(f"  Errors: {errors}/{len(runs)}")

except Exception as e:
    print(f"Note: {e}")
    print("(Set LANGCHAIN_API_KEY to access LangSmith data)")

## 7️⃣ LangSmith Prompt Hub

The LangSmith Prompt Hub lets you store, version, and share prompts:

```python
from langchain_classic import hub

# Pull a prompt from the hub
prompt = hub.pull("hwchase17/react")           # ReAct agent prompt
prompt = hub.pull("langchain-ai/rag-prompt")   # Standard RAG prompt

# Push your own prompt to the hub
hub.push("myorg/my-custom-prompt", my_prompt, new_repo_is_public=False)
```

## 8️⃣ Debugging Best Practices

```python
# 1. Enable verbose logging
from langchain.globals import set_verbose, set_debug
set_verbose(True)   # Shows chain steps
set_debug(True)     # Shows ALL inputs/outputs

# 2. Use callbacks for custom logging
from langchain_core.callbacks import StdOutCallbackHandler
chain.invoke(input, config={"callbacks": [StdOutCallbackHandler()]})

# 3. Inspect intermediate steps
chain_with_history = chain.with_config(return_only_outputs=False)
```

In [ ]:
# ============================================================
# Enable verbose debugging locally
# ============================================================
from langchain.globals import set_verbose

set_verbose(True)  # Show detailed chain execution

chain = (
    ChatPromptTemplate.from_template("Summarize: {text}")
    | llm | StrOutputParser()
)

result = chain.invoke({"text": "LangSmith is a platform for LLM observability."})

set_verbose(False)  # Turn off
print(f"\nResult: {result}")

## ✅ Module 11 Summary

You've learned:
- ✅ LangSmith setup and automatic tracing
- ✅ `@traceable` for custom function tracing
- ✅ Adding metadata and tags to runs
- ✅ Creating evaluation datasets
- ✅ LangChain built-in evaluators
- ✅ Custom LLM-as-judge evaluators
- ✅ Querying production metrics
- ✅ LangSmith Prompt Hub
- ✅ Verbose debugging

### 🚀 Next: [Module 12 — Production Patterns & Best Practices](12_Production_Best_Practices.ipynb)